In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib, numpy as np, pandas as pd, torch
CODE = "/content/drive/MyDrive/esci/code"; sys.path.insert(0, CODE)
import metrics, cross_encoder as C, train_cross_encoder as T
for m in (metrics, C, T): importlib.reload(m)
from metrics import evaluate

BASE = "/content/drive/MyDrive/esci"
df = pd.read_parquet(f"{BASE}/processed/rerank_pairs_v2.3.parquet")
qs = pd.read_parquet(f"{BASE}/dataset_v2.3/esci_electronics_v2.3_queries.parquet")[["query_id", "rule", "scope_group"]]
df = df.merge(qs, on="query_id", how="left")
assert df["rule"].notna().all()

# leak checks (test only used here, then dropped)
ids = {p: set(df.loc[df.part == p, "query_id"]) for p in ["train", "val", "test"]}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
norm = lambda s: " ".join(str(s).lower().split())
assert not (set(df.loc[df.part == "train", "query"].map(norm)) & set(df.loc[df.part == "val", "query"].map(norm)))
train = df[df.part == "train"].reset_index(drop=True)
val   = df[df.part == "val"].reset_index(drop=True)
del df

# M3 category stand-in: Rule A only (Rule B category comes from labelled products = leak)
CAT = {"mobile_computing": "Mobile & Computing", "components_accessories": "Components & Accessories",
       "home_electronics": "Home Electronics", "imaging": "Cameras & Imaging", "audio": "Audio",
       "computer_peripherals": "Computer Peripherals", "gaming": "Gaming", "smart_home": "Smart Home"}
for d in (train, val):
    d["category"] = np.where(d["rule"] == "A", d["scope_group"].map(CAT), None)
    d["q_raw"]         = d["query"].map(C._clean)
    d["q_raw_cat"]     = [C.query_to_text(q, category=c, use_rules=False) for q, c in zip(d["query"], d["category"])]
    d["q_focused"]     = [C.query_to_text(q) for q in d["query"]]
    d["q_focused_cat"] = [C.query_to_text(q, category=c) for q, c in zip(d["query"], d["category"])]
    prods = [C.esci_row_to_product(r) for r in d.to_dict("records")]
    for s in C.STYLES:
        d[f"text_{s}"] = [C.product_to_text(p, s) for p in prods]

QUERY_INPUTS = ["q_raw", "q_raw_cat", "q_focused", "q_focused_cat"]
print("product formats:", C.STYLES, "| same in trainer:", T.STYLES == C.STYLES)
print("train rows", len(train), "| val rows", len(val), "| GPU", torch.cuda.get_device_name(0))
print("val queries with category:", val.loc[val.category.notna(), "query_id"].nunique(), "of", val.query_id.nunique())
ex = val.drop_duplicates("query_id").sample(4, random_state=3)
print(ex[["query"] + QUERY_INPUTS].to_string())
print(val[["text_title", "text_key_value"]].head(2).to_string())

Mounted at /content/drive
product formats: ['title', 'title_brand', 'full', 'full_bullets', 'key_value'] | same in trainer: True
train rows 65852 | val rows 11741 | GPU Tesla T4
val queries with category: 495 of 656
                         query                     q_raw                                   q_raw_cat                 q_focused                               q_focused_cat
4488     galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2     galaxy watch active 2  [Mobile & Computing] galaxy watch active 2
736                a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv               a8g oled tv              [Home Electronics] a8g oled tv
1892                  aux cord                  aux cord                                    aux cord                  aux cord                                    aux cord
8364  overhead headphone wired  overhead headphone wired            [Audio] overhead headphone wired

In [ ]:
MODELS = {"minilm_l6":  "cross-encoder/ms-marco-MiniLM-L-6-v2",
          "minilm_l12": "cross-encoder/ms-marco-MiniLM-L-12-v2",
          "distilbert": "distilbert-base-uncased",
          "bert_base":  "bert-base-uncased"}
LOG = "s7_grid_val_log.csv"                  # same log as S7, so every run is in one table

def run_model(short, seed=42, accum=1, inputs=("q_focused_cat", "q_raw")):
    for qin in inputs:
        cfg = {**T.CFG, "query_col": qin, "styles": C.STYLES, "accum": accum, "seed": seed}
        name = f"{short}__{qin}" if seed == 42 else f"{short}__{qin}__s{seed}"
        print(f"\n===== {name} =====")
        T.train_run(MODELS[short], name, train, val, evaluate, BASE, cfg=cfg, log_name=LOG)

In [ ]:
run_model("minilm_l12")


===== minilm_l12__q_focused_cat =====
minilm_l12__q_focused_cat: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-12-v2


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4505, 3.1 min
       style  ndcg@10    mrr    p@5
       title   0.9238 0.9432 0.8460
 title_brand   0.9236 0.9392 0.8448
        full   0.9246 0.9420 0.8470
full_bullets   0.9287 0.9458 0.8488
   key_value   0.9257 0.9426 0.8466
  epoch 2 step 500/2058 loss 0.3299
  epoch 2 step 1000/2058 loss 0.3244
  epoch 2 step 1500/2058 loss 0.3207
  epoch 2 step 2000/2058 loss 0.3180


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3182, 3.0 min
       style  ndcg@10    mrr    p@5
       title   0.9225 0.9374 0.8470
 title_brand   0.9238 0.9378 0.8454
        full   0.9245 0.9373 0.8466
full_bullets   0.9278 0.9439 0.8460
   key_value   0.9249 0.9399 0.8457
  epoch 3 step 500/2058 loss 0.2840
  epoch 3 step 1000/2058 loss 0.2855
  epoch 3 step 1500/2058 loss 0.2855
  epoch 3 step 2000/2058 loss 0.2839


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.2844, 3.1 min
       style  ndcg@10    mrr    p@5
       title   0.9216 0.9371 0.8436
 title_brand   0.9220 0.9384 0.8439
        full   0.9232 0.9383 0.8460
full_bullets   0.9286 0.9456 0.8454
   key_value   0.9242 0.9377 0.8457


In [ ]:
run_model("distilbert")


===== distilbert__q_focused_cat =====
distilbert__q_focused_cat: starting at epoch 1 from distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4389, 4.0 min
       style  ndcg@10    mrr    p@5
       title   0.9169 0.9404 0.8354
 title_brand   0.9177 0.9377 0.8357
        full   0.9176 0.9363 0.8369
full_bullets   0.9165 0.9385 0.8381
   key_value   0.9183 0.9380 0.8378
  epoch 2 step 500/2058 loss 0.3281
  epoch 2 step 1000/2058 loss 0.3212
  epoch 2 step 1500/2058 loss 0.3184
  epoch 2 step 2000/2058 loss 0.3150


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3150, 4.0 min
       style  ndcg@10    mrr    p@5
       title   0.9188 0.9331 0.8387
 title_brand   0.9204 0.9338 0.8375
        full   0.9219 0.9364 0.8409
full_bullets   0.9202 0.9354 0.8424
   key_value   0.9206 0.9366 0.8402
  epoch 3 step 500/2058 loss 0.2666
  epoch 3 step 1000/2058 loss 0.2697
  epoch 3 step 1500/2058 loss 0.2700
  epoch 3 step 2000/2058 loss 0.2685


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.2688, 4.0 min
       style  ndcg@10    mrr    p@5
       title   0.9197 0.9340 0.8409
 title_brand   0.9230 0.9402 0.8415
        full   0.9230 0.9392 0.8436
full_bullets   0.9222 0.9388 0.8418
   key_value   0.9218 0.9398 0.8433


In [ ]:
run_model("bert_base")


===== bert_base__q_focused_cat =====
bert_base__q_focused_cat: starting at epoch 1 from bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4161, 7.9 min
       style  ndcg@10    mrr    p@5
       title   0.9235 0.9403 0.8451
 title_brand   0.9246 0.9417 0.8470
        full   0.9250 0.9430 0.8476
full_bullets   0.9256 0.9399 0.8436
   key_value   0.9238 0.9418 0.8473
  epoch 2 step 500/2058 loss 0.3030
  epoch 2 step 1000/2058 loss 0.2971
  epoch 2 step 1500/2058 loss 0.2950
  epoch 2 step 2000/2058 loss 0.2930


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.2931, 7.8 min
       style  ndcg@10    mrr    p@5
       title   0.9231 0.9387 0.8460
 title_brand   0.9232 0.9370 0.8448
        full   0.9245 0.9400 0.8457
full_bullets   0.9259 0.9462 0.8454
   key_value   0.9243 0.9388 0.8466
  epoch 3 step 500/2058 loss 0.2466
  epoch 3 step 1000/2058 loss 0.2462
  epoch 3 step 1500/2058 loss 0.2466
  epoch 3 step 2000/2058 loss 0.2446


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.2451, 7.8 min
       style  ndcg@10    mrr    p@5
       title   0.9248 0.9406 0.8476
 title_brand   0.9255 0.9387 0.8476
        full   0.9257 0.9383 0.8479
full_bullets   0.9280 0.9441 0.8470
   key_value   0.9263 0.9393 0.8454


In [ ]:
LOGDF = pd.read_csv(f"{BASE}/results/{LOG}")
LOGDF["short"] = LOGDF.run.str.extract(r"^([a-z0-9_]+?)__")[0]
LOGDF["qin"]   = LOGDF.run.str.extract(r"__(q_[a-z_]+?)(?:__s\d+)?$")[0]
LOGDF["seed"]  = LOGDF.run.str.extract(r"__s(\d+)$")[0].fillna("42").astype(int)
per_run = LOGDF.groupby(["short", "qin", "seed", "epoch"])[["ndcg@10", "mrr", "p@5", "cum_train_time_s"]] \
               .agg({"ndcg@10": "mean", "mrr": "mean", "p@5": "mean", "cum_train_time_s": "max"}).reset_index()
best = per_run.loc[per_run.groupby(["short", "qin", "seed"])["ndcg@10"].idxmax()]
tab = best[best.qin.isin(["q_focused_cat", "q_raw"])].groupby(["short", "qin"]) \
          .agg(ndcg=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"), mrr=("mrr", "mean"), p5=("p@5", "mean"),
               seeds=("seed", "count"), best_epoch=("epoch", "max"), train_min=("cum_train_time_s", lambda s: s.max()/60))
print(tab.round(4).to_string())

# per-format detail for focused_cat (seed 42, best epoch)
b42 = best[(best.qin == "q_focused_cat") & (best.seed == 42)]
det = LOGDF.merge(b42[["short", "qin", "seed", "epoch"]], on=["short", "qin", "seed", "epoch"])
print("\nper format (focused_cat, seed 42):\n",
      det.pivot_table(index="short", columns="style", values="ndcg@10").round(4).to_string())
best.to_csv(f"{BASE}/results/s8_best_epochs.csv", index=False)

                            ndcg  ndcg_std     mrr      p5  seeds  best_epoch  train_min
short      qin                                                                          
bert_base  q_focused_cat  0.9251       NaN  0.9415  0.8477      1           3    24.1883
           q_raw          0.9261       NaN  0.9402  0.8471      1           3    23.5167
distilbert q_focused_cat  0.9209       NaN  0.9382  0.8408      1           2     8.2117
           q_raw          0.9219       NaN  0.9384  0.8422      1           3    12.0383
minilm_l12 q_focused_cat  0.9301       NaN  0.9505  0.8483      1           2     6.1333
           q_raw          0.9253       NaN  0.9426  0.8466      1           1     3.0500
minilm_l6  q_focused_cat  0.9279    0.0010  0.9420  0.8489      3           3     5.2367
           q_raw          0.9277    0.0009  0.9415  0.8457      3           3     4.9983

per format (focused_cat, seed 42):
 style         full  full_bullets  key_value   title  title_brand
short   

In [ ]:
import os
print(sorted(os.listdir("/content/drive/MyDrive/esci/models"))[:20])
print(os.path.exists("/content/drive/MyDrive/esci/results/s7_grid_val_log.csv"))

['bert_base__q_focused_cat', 'bert_base__q_raw', 'distilbert__q_focused_cat', 'distilbert__q_raw', 'minilm_l12__q_focused_cat', 'minilm_l12__q_raw', 'minilm_l6__q_focused', 'minilm_l6__q_focused__s43', 'minilm_l6__q_focused__s44', 'minilm_l6__q_focused_cat', 'minilm_l6__q_focused_cat__s43', 'minilm_l6__q_focused_cat__s44', 'minilm_l6__q_raw', 'minilm_l6__q_raw__s43', 'minilm_l6__q_raw__s44', 'minilm_l6__q_raw_cat', 'minilm_l6__q_raw_cat__s43', 'minilm_l6__q_raw_cat__s44', 'minilm_l6_ft']
True


In [ ]:
run_model("minilm_l12", seed=43)     # focused_cat + raw
run_model("minilm_l12", seed=44)


===== minilm_l12__q_focused_cat__s43 =====
minilm_l12__q_focused_cat__s43: starting at epoch 1 from cross-encoder/ms-marco-MiniLM-L-12-v2


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

epoch 1 done: loss 0.4521, 2.9 min
       style  ndcg@10    mrr    p@5
       title   0.9236 0.9449 0.8448
 title_brand   0.9244 0.9424 0.8460
        full   0.9249 0.9428 0.8460
full_bullets   0.9308 0.9475 0.8482
   key_value   0.9249 0.9463 0.8454
  epoch 2 step 500/2058 loss 0.3347
  epoch 2 step 1000/2058 loss 0.3281
  epoch 2 step 1500/2058 loss 0.3240
  epoch 2 step 2000/2058 loss 0.3204


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2 done: loss 0.3194, 2.9 min
       style  ndcg@10    mrr    p@5
       title   0.9244 0.9384 0.8457
 title_brand   0.9239 0.9370 0.8466
        full   0.9246 0.9394 0.8479
full_bullets   0.9290 0.9465 0.8485
   key_value   0.9256 0.9409 0.8473
  epoch 3 step 500/2058 loss 0.2870
  epoch 3 step 1000/2058 loss 0.2889
  epoch 3 step 1500/2058 loss 0.2887
  epoch 3 step 2000/2058 loss 0.2867


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3 done: loss 0.2862, 2.9 min
       style  ndcg@10    mrr    p@5
       title   0.9224 0.9382 0.8436
 title_brand   0.9238 0.9399 0.8430
        full   0.9235 0.9394 0.8460
full_bullets   0.9289 0.9468 0.8470
   key_value   0.9243 0.9388 0.8454


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
from scipy.stats import wilcoxon

BASE = "/content/drive/MyDrive/esci"
LOG  = "s7_grid_val_log.csv"

# ---- rebuild the summary table (same as Cell 6)
LOGDF = pd.read_csv(f"{BASE}/results/{LOG}")
LOGDF["short"] = LOGDF.run.str.extract(r"^([a-z0-9_]+?)__")[0]
LOGDF["qin"]   = LOGDF.run.str.extract(r"__(q_[a-z_]+?)(?:__s\d+)?$")[0]
LOGDF["seed"]  = LOGDF.run.str.extract(r"__s(\d+)$")[0].fillna("42").astype(int)
per_run = LOGDF.groupby(["short", "qin", "seed", "epoch"]).agg(
    {"ndcg@10": "mean", "mrr": "mean", "p@5": "mean", "cum_train_time_s": "max"}).reset_index()
best = per_run.loc[per_run.groupby(["short", "qin", "seed"])["ndcg@10"].idxmax()]
tab = best[best.qin.isin(["q_focused_cat", "q_raw"])].groupby(["short", "qin"]).agg(
    ndcg=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"), mrr=("mrr", "mean"), p5=("p@5", "mean"),
    seeds=("seed", "count"), best_epochs=("epoch", lambda s: list(s)))
print(tab.round(4).to_string())

# ---- paired significance tests (per-query NDCG, averaged over seeds)
def perq_avg(short, qin):
    frames = []
    for _, r in best[(best.short == short) & (best.qin == qin)].iterrows():
        name = f"{short}__{qin}" if r.seed == 42 else f"{short}__{qin}__s{r.seed}"
        d = pd.read_csv(f"{BASE}/models/{name}/per_query_val_epoch_{int(r.epoch)}.csv")
        frames.append(d.groupby("query_id")["ndcg@10"].mean())
    return pd.concat(frames, axis=1).mean(axis=1), len(frames)

tests = [(("minilm_l12", "q_focused_cat"), ("minilm_l12", "q_raw")),
         (("minilm_l12", "q_focused_cat"), ("minilm_l6",  "q_focused_cat")),
         (("minilm_l12", "q_focused_cat"), ("bert_base",  "q_focused_cat"))]
print()
for (a_s, a_q), (b_s, b_q) in tests:
    xa, na = perq_avg(a_s, a_q); xb, nb = perq_avg(b_s, b_q)
    d = xa - xb.loc[xa.index]; nz = d[d != 0]
    print(f"{a_s}/{a_q} ({na} seeds) vs {b_s}/{b_q} ({nb} seeds): mean diff {d.mean():+.4f} | "
          f"better {int((d>0).sum())} / worse {int((d<0).sum())} | Wilcoxon p = {wilcoxon(nz).pvalue:.4f}")

best.to_csv(f"{BASE}/results/s8_best_epochs.csv", index=False)

Mounted at /content/drive
                            ndcg  ndcg_std     mrr      p5  seeds best_epochs
short      qin                                                               
bert_base  q_focused_cat  0.9251       NaN  0.9415  0.8477      1         [3]
           q_raw          0.9261       NaN  0.9402  0.8471      1         [3]
distilbert q_focused_cat  0.9209       NaN  0.9382  0.8408      1         [2]
           q_raw          0.9219       NaN  0.9384  0.8422      1         [3]
minilm_l12 q_focused_cat  0.9291    0.0014  0.9477  0.8492      3   [2, 1, 3]
           q_raw          0.9259    0.0007  0.9434  0.8476      3   [1, 3, 1]
minilm_l6  q_focused_cat  0.9279    0.0010  0.9420  0.8489      3   [3, 3, 2]
           q_raw          0.9277    0.0009  0.9415  0.8457      3   [3, 2, 2]

minilm_l12/q_focused_cat (3 seeds) vs minilm_l12/q_raw (3 seeds): mean diff +0.0032 | better 162 / worse 149 | Wilcoxon p = 0.1806
minilm_l12/q_focused_cat (3 seeds) vs minilm_l6/q_focused_cat 